# Split por Rótulos com Chunking (documento executável)

Segmenta os streams extraídos em classes (`square`, `front_to_back`, `side_to_side`, `shoot_to_goal`) e
**quebra intervalos longos em sub-eventos de duração máxima** (`CHUNK_MAX_SEC`). A lógica vive em
`dtssl.data.extract` (`annotations_frame`, `chunk_intervals`, `validate_intervals`, `split_dataset`,
`split_by_labels`); este notebook só chama as funções.

Estratégia de chunking: cada anotação de duração `D` vira `N = ⌈D / CHUNK_MAX_SEC⌉` sub-intervalos de
tamanho **igual** `D/N`. Sem sobreposição, sem descarte de resto.

**event_id**: `<label>_<annotation>_<chunk>` (ex.: `square_01_03` = label `square`, 1ª anotação, 3º chunk).

**Saídas** (`split_by_labels`):
- `splits/<label>/<event_id>/<stream>.parquet` — um diretório por sub-evento, com `timestamp_event = timestamp - t_start`
- `splits/all_<stream>_labeled.parquet` — tabela concatenada por stream com `label`, `event_id`, `source_idx`, `chunk_idx`
- `splits/labels_manifest.csv` — metadados (duração, linhas por stream, lineage)

Equivalente em linha de comando: `dtssl etl split data/extracted/2026-05-18_19-2-15 [labels.csv] [--chunk-s 10]`.


## 1. Setup

In [ ]:
from pathlib import Path

import pandas as pd

from dtssl.data.extract import (ANNOTATIONS, annotations_frame, chunk_intervals, read_dataset,
                                split_by_labels, validate_intervals)

# ------------------ CONFIG ------------------
EXTRACTED_DIR = Path('extracted/2026-05-18_19-2-15')   # troque pelo log
CHUNK_MAX_SEC = 10.0          # 10 s -> bom equilíbrio; troque pra 5.0 se quiser mais amostras
LABELS_CSV    = None          # None = anotações embutidas (ANNOTATIONS[<dataset>]); ou 'labels.csv' (label,t_start,t_end)
# --------------------------------------------
print('Lendo de:', EXTRACTED_DIR.resolve())
print(f'Chunk máximo: {CHUNK_MAX_SEC}s')
print('Datasets com anotações embutidas:', sorted(ANNOTATIONS))


## 2. Carregar os streams

In [ ]:
dfs = read_dataset(EXTRACTED_DIR)
for name, df in dfs.items():
    print(f'{name:>20s}: {len(df):>8d} linhas')


## 3. Anotações (intervalos brutos)

Tempos em segundos a partir de `t=0` do log. As anotações do log `2026-05-18_19-2-15` estão embutidas em
`dtssl.data.extract.ANNOTATIONS` (as mesmas do notebook original); para outro log, escreva um CSV
`label,t_start,t_end`.

In [ ]:
intervals_raw = annotations_frame(LABELS_CSV, dataset=EXTRACTED_DIR.name)
intervals_raw


## 4. Chunking — quebrar intervalos longos em sub-eventos iguais

In [ ]:
intervals = chunk_intervals(intervals_raw, CHUNK_MAX_SEC)
print(f'Anotações originais: {len(intervals_raw)}  ->  sub-eventos após chunking: {len(intervals)}')
summary = (intervals.groupby('label')
                    .agg(n_events=('event_id', 'count'), n_sources=('source_idx', 'nunique'),
                         total_sec=('duration', 'sum'), mean_sec=('duration', 'mean'),
                         min_sec=('duration', 'min'), max_sec=('duration', 'max'))
                    .round(2).sort_values('n_events', ascending=False))
summary


## 5. Validação (chunks dentro do log; sobreposições entre anotações só são registradas)

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format='%(message)s')
report = validate_intervals(intervals, dfs)
report['overlaps']


## 6. Split

In [ ]:
manifest = split_by_labels(EXTRACTED_DIR, intervals_raw, CHUNK_MAX_SEC)
print(f'{len(manifest)} sub-eventos salvos em {(EXTRACTED_DIR / "splits").resolve()}')
manifest.head(10)


## 7. Sanidade — quantas linhas cada stream recebeu por label?

In [ ]:
row_cols = [c for c in manifest.columns if c.endswith('_rows')]
by_label = manifest.groupby('label')[row_cols].sum()
by_label.insert(0, 'n_events', manifest.groupby('label').size())
thin = manifest[manifest['ball_rows'] < 30][['event_id', 't_start', 't_end', 'duration', 'ball_rows']]
print('chunks com < 30 amostras de bola (provável buraco no log):', len(thin))
by_label


## 8. Receitas de uso

```python
big = pd.read_parquet('extracted/<log>/splits/all_processed_robots_labeled.parquet')
chutes = big[big['label'] == 'shoot_to_goal']
for ev_id, ev_df in chutes.groupby('event_id'):
    ...  # ev_df.timestamp_event começa em 0 -> séries comparáveis

# carregar um evento no schema canônico em SI (m, m/s, rad; grade 1/60 s)
from dtssl.data.loader import load_event
seg = load_event('extracted/<log>/splits/square/square_01_03/commands.parquet',
                 'extracted/<log>/splits/square/square_01_03/processed_robots.parquet',
                 team='allies', robot_id=3)
```